In [2]:
from pathlib import Path
import pandas as pd

RAW_INPUT = Path("/Users/jschmidt3/NMDpredictionmodel/Model/TrunCat/data/TOPMed_stopgain.csv")
CV_PATH = Path("/Users/jschmidt3/NMDpredictionmodel/Model/TrunCat/results/cv_predictions.csv")
OUT_PATH = Path("/Users/jschmidt3/NMDpredictionmodel/Model/TrunCat/results/cv_predictions_with_ids.csv")

TARGET = "NMD.ESCAPEE"

df = pd.read_csv(RAW_INPUT, low_memory=False)
cv = pd.read_csv(CV_PATH)

y_full = df[TARGET].map({
    "TRUE": 1, "FALSE": 0,
    "True": 1, "False": 0,
    True: 1, False: 0,
    1: 1, 0: 0,
}).astype("float")

mask = y_full.notna()

df_valid = df.loc[mask].reset_index(drop=True).copy()
df_valid["cv_index"] = range(len(df_valid))

# TrunCat cv_predictions has no index, so create one from row order
cv = cv.reset_index().rename(columns={"index": "cv_index"})

# Standardize names to match TrunKitten-style notebook inputs
cv = cv.rename(columns={
    "true_label": "y_true",
    "predicted_prob": "oof_prob",
    "predicted_class": "oof_pred_at_0.5",
})

candidate_id_cols = [
    "cv_index",
    "variantID", "variant_id", "key",
    "contig", "position", "CHROM", "POS",
    "refAllele", "altAllele", "REF_ALLELE", "ALT_ALLELE", "REF", "ALT",
    "txnames", "GENE_ID", "Gene_ID", "gene", "hgnc_symbol",
]

id_cols = [c for c in candidate_id_cols if c in df_valid.columns]
print("ID columns found:", id_cols)

cv_with_ids = cv.merge(
    df_valid[id_cols],
    on="cv_index",
    how="left",
    validate="one_to_one",
)

print("Raw input:", df.shape)
print("Valid target rows:", df_valid.shape)
print("CV predictions:", cv.shape)
print("CV + IDs:", cv_with_ids.shape)

label_match = (
    cv_with_ids["y_true"].astype(int).values
    == y_full.loc[mask].astype(int).values
).mean()

print(f"Label match between cv y_true and reconstructed target: {label_match:.6f}")

assert len(cv_with_ids) == len(cv), "Merge changed row count."
assert label_match == 1.0, "Labels do not match; row-order mapping is not safe."

cv_with_ids.to_csv(OUT_PATH, index=False)
print("Wrote:", OUT_PATH)

ID columns found: ['cv_index', 'variantID', 'key', 'contig', 'position', 'CHROM', 'POS', 'refAllele', 'altAllele', 'REF_ALLELE', 'ALT_ALLELE', 'txnames', 'GENE_ID', 'gene', 'hgnc_symbol']
Raw input: (6019, 1034)
Valid target rows: (5749, 1035)
CV predictions: (5749, 4)
CV + IDs: (5749, 18)
Label match between cv y_true and reconstructed target: 1.000000
Wrote: /Users/jschmidt3/NMDpredictionmodel/Model/TrunCat/results/cv_predictions_with_ids.csv
